# Samian ware: production centres and their kiln regions

This local Jupyter notebook queries the NFDI4Objects Knowledge Graph for Samian-ware **production centres**, retrieves their geographic coordinates and their assignment to higher-level **kiln regions**, and visualises the result on an interactive map.

A companion browser-executable notebook, `n4okg-samian-production-centres-live.qmd`, runs the same pipeline via Pyodide and Leaflet directly in the browser for readers who prefer a server-less environment.


## Requirements

```bash
pip install SPARQLWrapper pandas matplotlib folium
```


## About this notebook

Samian ware (terra sigillata) was produced across the Roman Empire between the 1st century BCE and the 3rd century CE at a set of well-known workshop locations — La Graufesenque, Lezoux, Rheinzabern, and many others. The NFDI4Objects Knowledge Graph models each workshop as a `lado:ProductionCentre` and groups centres into higher-level `KilnRegion` clusters (e.g. *South Gaulish*, *Central Gaulish*).

### Why this dataset?

Samian-ware production is a particularly good pedagogical dataset for geographic SPARQL visualisation: the number of centres is small enough (around 100) to plot as individual markers without clutter, the categorical grouping by kiln region gives a natural colour dimension, and the research tradition is mature enough that workshop locations and their clustering are well established.

### What you'll learn

- how to query a GeoSPARQL-typed `asWKT` literal from a Wikibase-style knowledge graph
- how to parse WKT point literals robustly across endpoints
- how to build a categorical `folium` marker map directly from a pandas DataFrame

### Data-context notes

- `ProductionCentre` → `KilnRegion` is modelled via `lado:clusteredAs`
- geometries use `geosparql:hasGeometry` / `geosparql:asWKT` — a two-step indirection (the `asWKT` is attached to the geometry node, not directly to the centre)
- WKT literals on this endpoint use `POINT(lon lat)` (uppercase); the parser below handles both casings defensively
- one centre may in principle have more than one kiln-region label attached; the code de-duplicates to one row per centre for the map

### Tooling notes

Locally we use `SPARQLWrapper` (cleanest Python API for SPARQL endpoints) and `folium` (leaflet.js bindings that work seamlessly in Jupyter). Both have no Pyodide-compatible counterpart — the browser twin uses `pyodide.http.pyfetch` and hand-written Leaflet instead.


## Step 1 — Define the SPARQL query

We retrieve every `ProductionCentre`, its label, its WKT geometry, and the label of its associated `KilnRegion`. The join across the geometry blank node (`?item_geom`) is a common pattern in GeoSPARQL data: the geometry literal is never attached directly to the feature, always via an intermediate node.


In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

USER_AGENT = "OER-Quarto-Notebook/1.0 (https://n4o-rse.github.io/oer-ta2-koeln/)"
SPARQL_ENDPOINT = "https://graph.nfdi4objects.net/api/sparql"

SPARQL_QUERY = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX lado: <http://archaeology.link/ontology#>
SELECT ?item ?itemLabel ?geo ?kilnregion WHERE {
  GRAPH <https://graph.nfdi4objects.net/collection/8> {
    ?item <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://archaeology.link/ontology#ProductionCentre> .
    ?item <http://www.opengis.net/ont/geosparql#hasGeometry> ?item_geom .
    ?item_geom <http://www.opengis.net/ont/geosparql#asWKT> ?geo .
    ?item rdfs:label ?itemLabel .
    ?item lado:clusteredAs ?kr .
    ?kr rdfs:label ?kilnregion .
  }
}
"""

sparql = SPARQLWrapper(SPARQL_ENDPOINT, agent=USER_AGENT)
sparql.setQuery(SPARQL_QUERY)
sparql.setReturnFormat(JSON)
print("✓ Helpers defined. Run the next cell to load the data.")


## Step 2 — Load the data

The bindings from a SPARQL JSON result are flattened into a DataFrame. The WKT literal `POINT(lon lat)` is split into separate `latitude` and `longitude` columns using a case-insensitive regex parser — the same endpoint occasionally uses mixed casing across different collections, so we avoid brittle string slicing.


In [ ]:
import re

_WKT_POINT_RE = re.compile(r"point\s*\(\s*([-+\d.eE]+)\s+([-+\d.eE]+)\s*\)",
                           re.IGNORECASE)

def parse_wkt_point(wkt):
    """Parse 'POINT(lon lat)' (any case) into (lat, lon) ready for mapping libraries."""
    if not wkt:
        return (None, None)
    m = _WKT_POINT_RE.search(wkt)
    if not m:
        return (None, None)
    try:
        lon, lat = float(m.group(1)), float(m.group(2))
        return (lat, lon)
    except ValueError:
        return (None, None)


In [ ]:
bindings = sparql.query().convert()["results"]["bindings"]

rows = []
for b in bindings:
    lat, lon = parse_wkt_point(b.get("geo", {}).get("value"))
    rows.append({
        "item": b["item"]["value"],
        "itemLabel": b["itemLabel"]["value"],
        "kilnregion": b["kilnregion"]["value"],
        "latitude": lat,
        "longitude": lon,
    })

df = pd.DataFrame(rows)
df = df.dropna(subset=["latitude", "longitude"])
df = df.drop_duplicates(subset="item", keep="first").reset_index(drop=True)

print(f"Loaded {len(df)} production centres across "
      f"{df['kilnregion'].nunique()} kiln regions")
df.head()


## Step 3a — Centres per kiln region (sanity check)

Before the map, a simple bar chart confirms the data shape and gives a numerical anchor for the spatial view that follows. Very uneven bars (one region with many centres, several with just one or two) are typical for Samian-ware production: research attention and survival bias favour the large South-Gaulish and Central-Gaulish clusters.


In [ ]:
import matplotlib.pyplot as plt

counts = df["kilnregion"].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, max(3, 0.35 * len(counts))))
ax.barh(counts.index, counts.values, color="#4c72b0")
ax.set_xlabel("Number of production centres")
ax.set_title("Samian-ware production centres per kiln region")
for i, v in enumerate(counts.values):
    ax.text(v + 0.5, i, str(int(v)), va="center")
plt.tight_layout()
plt.show()


## Step 3b — Map of production centres

Each centre becomes a `CircleMarker` on a `folium` map. Markers are coloured by kiln region, and a `LayerControl` is added so each region can be toggled individually. Four basemaps are available via the same control.


In [ ]:
import folium

PALETTE = [
    "#e6194B", "#3cb44b", "#4363d8", "#f58231", "#911eb4",
    "#42d4f4", "#f032e6", "#469990", "#9A6324", "#800000",
    "#808000", "#000075", "#e91e63", "#607d8b", "#795548",
]
regions = sorted(df["kilnregion"].unique())
region_colors = {r: PALETTE[i % len(PALETTE)] for i, r in enumerate(regions)}

lat_c = float(df["latitude"].mean())
lon_c = float(df["longitude"].mean())

m = folium.Map(location=[lat_c, lon_c], zoom_start=4, tiles=None)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer(
    "https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png",
    name="OpenStreetMap.HOT", attr="© OpenStreetMap contributors, HOT style"
).add_to(m)
folium.TileLayer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Esri.WorldImagery", attr="Tiles © Esri — Source: Esri, Maxar, Earthstar Geographics"
).add_to(m)
folium.TileLayer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Terrain_Base/MapServer/tile/{z}/{y}/{x}",
    name="Esri.WorldTerrain", attr="Tiles © Esri — Source: USGS, Esri, TANA, DeLorme, NPS", max_zoom=13
).add_to(m)

region_groups = {r: folium.FeatureGroup(name=r).add_to(m) for r in regions}

for _, row in df.iterrows():
    popup_html = (
        f"<b>{row['itemLabel']}</b><br>"
        f"<i>{row['kilnregion']}</i><br>"
        f'<a href="{row["item"]}" target="_blank">View in N4O KG</a>'
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color=region_colors[row["kilnregion"]],
        weight=1,
        fill=True,
        fill_color=region_colors[row["kilnregion"]],
        fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=300),
    ).add_to(region_groups[row["kilnregion"]])

folium.LayerControl(collapsed=True).add_to(m)
m


## Step 4 — Explore

The `df` DataFrame stays in scope — modify the cell below to filter, aggregate, or re-join however you like.


In [ ]:
# Hier anpassen: filter by kiln region, list centres, sort by latitude…
df[df["kilnregion"].str.contains("Gaul", case=False, na=False)] \
    .sort_values("latitude", ascending=False) \
    .head(10)


---

*Part of an Open Educational Resource series on knowledge graphs and linked open data, produced in the context of [NFDI4Objects](https://www.nfdi4objects.net/).*
